# 🔍 기능 3. Research Gap Analyzer - pgvector 문헌 검색 및 개별 논문 분석 (기초)

본 가이드는 **Paper Agent** 프로젝트의 **대규모 문헌 비교 분석기(Research Gap Analyzer)** 기능 중 **pgvector 기반 문헌 RAG 탐색, ArXiv ID 기준 중복 제거 및 청크 병합, gpt-4o-mini 구조화 출력을 통한 개별 논문 분석**의 핵심 동작 원리를 다룹니다.

학습용으로 불필요한 예외 처리 및 UI 렌더링 코드를 최소화하여, **실제 대규모 백그라운드 배치 연산의 1~2단계 핵심 구현 코드**를 간결하게 확인할 수 있도록 구성되었습니다.

---

## 📌 기초 학습 핵심 포인트
1. **백엔드 환경 설정 로드**
   - `backend/.env` 환경 변수와 패키지 경로를 주입합니다.
2. **질의어의 3072차원 변환**
   - `text-embedding-3-large` API를 통해 문헌 분석용 키워드를 임베딩 벡터로 얻는 원리를 파악합니다.
3. **pgvector RAG 검색 및 중복 제거**
   - `cs_embeddings` 등 도메인 컬렉션에서 코사인 거리 기준 유사 청크를 찾고, 논문 식별자(arXiv ID) 기준 중복 제거 및 텍스트 단락 병합 방법을 이해합니다.
4. **개별 논문 구조화 분석**
   - `gpt-4o-mini`와 `with_structured_output` API를 활용하여 각 논문의 핵심 해결점(`problems_solved`)과 한계점(`limitations`)을 텍스트 원문 인용(`source_quote`)과 함께 정교하게 추출하는 메커니즘을 배웁니다.

## 1단계. 백엔드 환경 설정 및 모듈 로드

노트북 실행 위치를 파악하고 백엔드 패키지 경로를 주입한 뒤, 필수 환경변수들을 로드합니다.

In [1]:
import os
import sys
import warnings
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

# 현재 notebooks/research_gap/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

# 백엔드 .env 로드
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정 로드 완료: {env_path}")

openai_key = os.getenv("OPENAI_API_KEY", "")
database_url = os.getenv("DATABASE_URL", "")

print(f"🔑 OpenAI API Key 설정 여부: {'설정됨(Yes)' if openai_key else '설정안됨(No)'}")
print(f"🗄️ Database URL: {database_url[:35]}..." if database_url else "🗄️ Database URL: 설정안됨")

✅ 백엔드 환경 설정 로드 완료: /Users/pileuszu/Repos/bist-mini-2/backend/.env
🔑 OpenAI API Key 설정 여부: 설정됨(Yes)
🗄️ Database URL: postgresql+asyncpg://postgres:postg...


## 2단계. Query Embedding 생성

사용자의 검색 주제(예: "attention mechanism optimization")를 3072차원의 쿼리 벡터로 변환합니다.

In [2]:
from langchain_openai import OpenAIEmbeddings

query = "low cost transformer training"

if not openai_key:
    query_vector = [0.0] * 3072
    print(f"🚨 OPENAI_API_KEY가 없습니다. 가상 벡터 생성 (차원: {len(query_vector)})")
else:
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-large",
        api_key=openai_key
    )
    query_vector = embeddings.embed_query(query)
    print(f"✅ 질의어 임베딩 변환 완료! 차원: {len(query_vector)} 차원")
    print(f"📊 벡터 샘플 (앞 5개 요소): {query_vector[:5]}")

✅ 질의어 임베딩 변환 완료! 차원: 3072 차원
📊 벡터 샘플 (앞 5개 요소): [-0.016021728515625, 0.039031982421875, -0.0091705322265625, 0.0163726806640625, 0.00582122802734375]


## 3단계. pgvector 유사도 검색, 중복 제거 및 청크 병합

데이터베이스에서 검색 쿼리 벡터와 코사인 유사도가 가장 높은 상위 문서 청크를 찾은 뒤,
동일 논문(arXiv ID 기준)에서 나온 여러 개의 세그먼트 청크를 하나로 묶고 중복을 제거하여 **상위 4개 대표 문헌 목록**을 완성하는 로직을 실습합니다.

In [3]:
# --- Mock 검색 데이터 정의 ---
# 실제 DB와 연동하지 않더라도 알고리즘 동작을 이해할 수 있도록 예제를 제공합니다.
mock_search_raw_results = [
    {"doc_id": "2301.0001", "title": "FlashAttention: Fast Attention with IO-Awareness", "text_chunk": "We propose FlashAttention to reduce memory accesses.", "score": 0.89},
    {"doc_id": "2301.0001", "title": "FlashAttention: Fast Attention with IO-Awareness", "text_chunk": "By blocking, we compute exact attention in fewer HBM reads.", "score": 0.88},
    {"doc_id": "2402.0002", "title": "Mamba: Linear-Time Sequence Modeling", "text_chunk": "Mamba establishes linear scaling with sequence length.", "score": 0.81},
    {"doc_id": "2305.0003", "title": "QLoRA: Efficient Finetuning of Quantized LLMs", "text_chunk": "QLoRA reduces finetuning memory via 4-bit NormalFloat.", "score": 0.78},
    {"doc_id": "2402.0002", "title": "Mamba: Linear-Time Sequence Modeling", "text_chunk": "We introduce selective state space models to address attention limits.", "score": 0.77}
]

def process_rag_deduplication(raw_results: list[dict], max_papers: int = 4) -> list[dict]:
    """백엔드 api/v1/research_gap/services.py와 동일한 중복 제거 및 청크 병합 알고리즘입니다."""
    temp_papers = {}
    for r in raw_results:
        arxiv_id = r["doc_id"]
        title = r["title"]
        content = r["text_chunk"]
        similarity = r["score"]
        
        if arxiv_id not in temp_papers:
            temp_papers[arxiv_id] = {
                "arxiv_id": arxiv_id,
               "title": title,
               "similarity": similarity,
               "chunks": [content]
            }
        else:
            chunks = temp_papers[arxiv_id]["chunks"]
            if content not in chunks:
                chunks.append(content)
                
    # 결과 정렬 및 최종 4개 제한
    papers_list = []
    for arxiv_id, p_info in temp_papers.items():
        if len(papers_list) >= max_papers:
            break
        joined_content = "\n\n".join(p_info["chunks"])
        papers_list.append({
            "arxiv_id": p_info["arxiv_id"],
            "title": p_info["title"],
            "similarity": p_info["similarity"],
            "content": joined_content
        })
    return papers_list

# 중복 제거 및 결합 실행
papers_processed = process_rag_deduplication(mock_search_raw_results)
print(f"📋 중복 제거 후 최종 분석 후보 수: {len(papers_processed)} 개 (원문 청크 5개 -> 병합 논문 {len(papers_processed)}개)")
for idx, p in enumerate(papers_processed, 1):
    print(f"{idx}. [{p['arxiv_id']}] {p['title']} (유사도: {p['similarity']})")
    print(f"   - 결합된 분석 본문: {p['content']}")

📋 중복 제거 후 최종 분석 후보 수: 3 개 (원문 청크 5개 -> 병합 논문 3개)
1. [2301.0001] FlashAttention: Fast Attention with IO-Awareness (유사도: 0.89)
   - 결합된 분석 본문: We propose FlashAttention to reduce memory accesses.

By blocking, we compute exact attention in fewer HBM reads.
2. [2402.0002] Mamba: Linear-Time Sequence Modeling (유사도: 0.81)
   - 결합된 분석 본문: Mamba establishes linear scaling with sequence length.

We introduce selective state space models to address attention limits.
3. [2305.0003] QLoRA: Efficient Finetuning of Quantized LLMs (유사도: 0.78)
   - 결합된 분석 본문: QLoRA reduces finetuning memory via 4-bit NormalFloat.


## 4단계. LLM을 활용한 개별 논문 구조화 분석

각 논문 본문으로부터 해결한 문제(`problems_solved`)와 한계점(`limitations`)을 각각 최대 2개 추출합니다.
이때 LLM이 왜곡하지 않고 본문에서 한 단어도 바꾸지 않은 원문 인용구(`source_quote`)를 매핑하여 Pydantic 모델(`PaperAnalysisResult`) 형태로 구조화 출력을 생성합니다.

In [4]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# --- Pydantic 스키마 선언 ---
class ExtractionItem(BaseModel):
    summary: str = Field(description="1-sentence summary of the extracted point (problem solved or limitation) in English")
    source_quote: str = Field(description="The exact verbatim sentence or paragraph from the provided content supporting this point")

class PaperAnalysisResult(BaseModel):
    arxiv_id: str = Field(description="The arXiv ID of the paper")
    title: str = Field(description="The title of the paper")
    problems_solved: list[ExtractionItem] = Field(description="List of at most 2 problems solved or methodologies proposed")
    limitations: list[ExtractionItem] = Field(description="List of at most 2 limitations, assumptions, or gaps identified")

# --- 분석 체인 ---
async def analyze_paper_abstract(paper_id: str, title: str, content: str) -> PaperAnalysisResult:
    if not openai_key:
        return PaperAnalysisResult(
            arxiv_id=paper_id,
            title=title,
            problems_solved=[ExtractionItem(summary="Reduces memory access overhead", source_quote="We propose FlashAttention to reduce memory accesses.")],
            limitations=[ExtractionItem(summary="Requires custom GPU kernels", source_quote="requires specialized CUDA development.")]
            
        )

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_key, temperature=0)
    structured_llm = llm.with_structured_output(PaperAnalysisResult)

    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are an academic researcher analyzing a scientific paper abstract.\n"
            "Extract the problems solved (or core methodologies proposed) and the limitations (or future gaps) "
            "discussed in the provided text.\n"
            "You must extract at most 2 problems solved and at most 2 limitations per paper.\n\n"
            "For each extracted item, you must also locate and extract the verbatim sentence or paragraph from the provided content "
            "that directly supports this claim, and place it in the 'source_quote' field of the item. "
            "Do not paraphrase or rewrite the 'source_quote' in any way; copy it exactly as it appears in the text.\n\n"
            "Response must be in English and structured in the requested format."
        )),
        ("user", "Title: {title}\nArXiv ID: {arxiv_id}\n\nContent:\n{content}")
    ])

    chain = prompt | structured_llm
    result = await chain.ainvoke({
        "title": title,
        "arxiv_id": paper_id,
        "content": content
    })
    
    # [LangChain Structured Output 타입 검증] 규칙 준수
    if not isinstance(result, PaperAnalysisResult):
        raise TypeError(f"Expected PaperAnalysisResult, got {type(result)}")
        
    return result

# 실행 테스트
analysis = await analyze_paper_abstract(
    "2301.0001", 
    "FlashAttention: Fast Attention with IO-Awareness", 
    "We propose FlashAttention to reduce memory accesses. Memory bound is the main bottleneck. FlashAttention computes exact attention faster but requires specialized CUDA development."
)
print(f"📄 논문 분석 완료: {analysis.title}")
print("   [해결 문제]")
for item in analysis.problems_solved:
    print(f"     - {item.summary} (원문: '{item.source_quote}')")
print("   [한계점]")
for item in analysis.limitations:
    print(f"     - {item.summary} (원문: '{item.source_quote}')")

📄 논문 분석 완료: FlashAttention: Fast Attention with IO-Awareness
   [해결 문제]
     - FlashAttention reduces memory accesses, addressing the memory bound bottleneck in attention computation. (원문: 'We propose FlashAttention to reduce memory accesses. Memory bound is the main bottleneck.')
     - FlashAttention computes exact attention faster than previous methods. (원문: 'FlashAttention computes exact attention faster but requires specialized CUDA development.')
   [한계점]
     - FlashAttention requires specialized CUDA development, which may limit its accessibility. (원문: 'FlashAttention computes exact attention faster but requires specialized CUDA development.')
